# Export post-fit $z$-expansion parameters to LaTeX

This notebook reloads the selected PROfit chains and generates copy-ready paper text containing the central values and posterior covariance matrices. Dipole fits are excluded.

Use the controls at the bottom and click **Reload fits and generate LaTeX** after changing a fit input or replacing a ROOT chain.

**Spline-factorization correction.** Every posterior quantity below is computed with the importance weights $w_k=\exp(+\Delta\chi^2_{\rm data}(\eta^{(k)})/2)$ that correct PROfit's multiplicative combination of the one-dimensional PCA splines to the exact z-expansion response (`python/scripts/spline_reweighting.py`, grids from `12_spline_factorization_validation.ipynb`, demonstration in `13_spline_reweighting_comparison.ipynb`). The weights are applied inside `postfit_physical_parameters.load_fit` and propagate to the summary tables, covariances, credible bands and corner plots; the effective sample size after reweighting is printed when each chain is loaded. The correction is negligible for the LQCD-constrained and Gaussian MINERvA priors and matters only for the uniform-prior fits.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display
import numpy as np

start = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (path / 'python' / 'scripts' for path in (start, *start.parents)
     if (path / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate python/scripts/postfit_physical_parameters.py')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from postfit_physical_parameters import SPECS, load_fit
from spline_reweighting import describe_ess, weighted_covariance, weighted_quantile

In [ ]:
SUITES = {
    'NuWro': 'nuwro_fit_results',
    'Open data': 'opendata_fit_results',
    'Asimov': 'asimov_fit_results',
}

# z-expansion fits only. Their ordering is also the output order.
FULL_COVARIANCE_FITS = {
    r'LQCD $k_{\max}=6$': 'lqcd_k6',
    r'LQCD $k_{\max}=7$': 'lqcd_k7',
    r'MINERvA $k_{\max}=6$': 'minerva_k6',
    r'MINERvA $k_{\max}=6$ + nuisances': 'minerva_k6_nuisance',
    r'MINERvA $k_{\max}=6$ uniform': 'minerva_k6_uniform',
    r'MINERvA $k_{\max}=6$ uniform + nuisances': 'minerva_k6_uniform_nuisance',
    r'MINERvA $k_{\max}=7$': 'minerva_k7',
    r'MINERvA $k_{\max}=8$': 'minerva_k8',
    r'MINERvA + LQCD $k_{\max}=6$': 'minerva_lqcd_k6',
    r'MINERvA + LQCD $k_{\max}=7$': 'minerva_lqcd_k7',
    r'MINERvA + LQCD $k_{\max}=6$ + nuisances': 'minerva_lqcd_k6_nuisance',
}

FIT_LATEX = {
    'lqcd_k6': r'\text{LQCD},\,k_{\max}=6',
    'lqcd_k7': r'\text{LQCD},\,k_{\max}=7',
    'minerva_k6': r'\text{MINERvA},\,k_{\max}=6',
    'minerva_k6_nuisance': r'\text{MINERvA},\,k_{\max}=6\;\text{with fitted nuisances}',
    'minerva_k6_uniform': r'\text{MINERvA},\,k_{\max}=6\;\text{uniform}',
    'minerva_k6_uniform_nuisance': r'\text{MINERvA},\,k_{\max}=6\;\text{uniform with fitted nuisances}',
    'minerva_k7': r'\text{MINERvA},\,k_{\max}=7',
    'minerva_k8': r'\text{MINERvA},\,k_{\max}=8',
    'minerva_lqcd_k6': r'\text{MINERvA + LQCD},\,k_{\max}=6',
    'minerva_lqcd_k7': r'\text{MINERvA + LQCD},\,k_{\max}=7',
    'minerva_lqcd_k6_nuisance': r'\text{MINERvA + LQCD}\;\text{with fitted nuisances}',
}

specs = {spec.key: spec for spec in SPECS}

In [ ]:
def decimal_latex(value, precision):
    value = 0.0 if abs(value) < 0.5 * 10**(-precision) else value
    return f'{value:.{precision}f}'


def signed_latex(value, precision):
    value = 0.0 if abs(value) < 0.5 * 10**(-precision) else value
    prefix = '' if value < 0 else r'\phantom{-}'
    return prefix + f'{value:.{precision}f}'


def central_values(result, convention, independent=True):
    indices = result['joint_indices'] if independent else np.arange(len(result['names']))
    samples = result['joint_samples'] if independent else result['samples']
    # Importance-weighted statistics (spline-factorization correction).
    if convention == 'Posterior mean':
        return np.average(samples, weights=result['weights'], axis=0)
    if convention == 'Posterior median':
        return weighted_quantile(samples, result['weights'], 0.5)
    if convention == 'Profile best fit':
        return result['profile'][indices]
    raise ValueError(f'Unknown central-value convention: {convention}')


def fit_block(
    dataset_label, key, result, convention, precision,
    covariance_precision, full_precision,
):
    names = result['joint_names']
    central = central_values(result, convention)
    covariance = weighted_covariance(result['joint_samples'], result['weights'])
    full_central = central_values(result, convention, independent=False)
    parameter_list = (
        rf'a_{{1}}, \ldots, a_{{{len(names)}}}' if len(names) > 2
        else ', '.join(rf'a_{{{name[1:]}}}' for name in names)
    )
    central_list = ', '.join(
        decimal_latex(value, precision) for value in central
    )
    full_values = [signed_latex(value, full_precision) for value in full_central]
    full_rows = []
    for start in range(0, len(full_values), 3):
        group = full_values[start:start + 3]
        suffix = ',' if start + 3 < len(full_values) else r'\big).'
        prefix = r'\big(&' if start == 0 else '&'
        full_rows.append(prefix + ','.join(group) + suffix)
    full_central_rows = (r'\\' + '\n               ').join(full_rows)
    covariance_rows = (' ' + r'\\' + '\n').join(
        ' & '.join(signed_latex(value, covariance_precision) for value in row)
        for row in covariance
    )
    fit_name = next(label for label, fit_key in FULL_COVARIANCE_FITS.items()
                    if fit_key == key)
    return rf'''% {dataset_label}: {fit_name}
we find the fitted parameters to be
\begin{{align}}
    \begin{{pmatrix}}
        {parameter_list}
    \end{{pmatrix}}
    = \begin{{pmatrix}}
        {central_list}
    \end{{pmatrix}},
\end{{align}}
with covariance matrix
\begin{{equation}}
    \begin{{aligned}}
        \begin{{pmatrix}}
        {covariance_rows}
        \end{{pmatrix}},
    \end{{aligned}}
\end{{equation}}
and the full precision coefficients $(a_0,\ldots,a_{len(full_central) - 1})$ are
\begin{{align}}
    \begin{{aligned}}
        {full_central_rows}
    \end{{aligned}}
\end{{align}}'''.strip()


def generate_latex(
    dataset_labels, fit_keys, convention, precision, covariance_precision,
    full_precision, burn_in, thin,
):
    blocks = []
    messages = []
    for dataset_label in dataset_labels:
        suite = SUITES[dataset_label]
        blocks.append(f'% ===== {dataset_label} =====')
        for key in fit_keys:
            result = load_fit(
                specs[key], suite, burn_in=burn_in, thin=thin
            )
            if result is None:
                messages.append(
                    f'Skipped {dataset_label} / {key}: no unique ROOT file.'
                )
                continue
            blocks.append(
                fit_block(
                    dataset_label, key, result, convention, precision,
                    covariance_precision, full_precision,
                )
            )
            messages.append(
                f'Loaded {dataset_label} / {key}: '
                f'{len(result["samples"]):,} posterior samples; '
                + describe_ess(result['ess'], len(result['samples']))
            )
    return '\n\n'.join(blocks), messages

## Interactive exporter

In [ ]:
dataset_widget = widgets.SelectMultiple(
    options=list(SUITES), value=tuple(SUITES), description='Datasets',
    rows=len(SUITES), layout=widgets.Layout(width='430px'),
)
fit_widget = widgets.SelectMultiple(
    options=[(label, key) for label, key in FULL_COVARIANCE_FITS.items()],
    value=tuple(FULL_COVARIANCE_FITS.values()), description='Fits',
    rows=len(FULL_COVARIANCE_FITS),
    layout=widgets.Layout(width='430px'),
)
central_widget = widgets.Dropdown(
    options=('Posterior mean', 'Posterior median', 'Profile best fit'),
    value='Posterior mean', description='Central value',
    layout=widgets.Layout(width='430px'),
)
precision_widget = widgets.BoundedIntText(
    value=2, min=0, max=10, description='Fit precision',
    layout=widgets.Layout(width='220px'),
)
covariance_precision_widget = widgets.BoundedIntText(
    value=8, min=1, max=16, description='Cov precision',
    layout=widgets.Layout(width='220px'),
)
full_precision_widget = widgets.BoundedIntText(
    value=8, min=2, max=16, description='Full precision',
    layout=widgets.Layout(width='220px'),
)
burn_in_widget = widgets.BoundedIntText(
    value=0, min=0, max=10_000_000, description='Burn in',
    layout=widgets.Layout(width='220px'),
)
thin_widget = widgets.BoundedIntText(
    value=1, min=1, max=100_000, description='Thin',
    layout=widgets.Layout(width='220px'),
)
generate_button = widgets.Button(
    description='Reload fits and generate LaTeX', icon='refresh',
    button_style='primary', layout=widgets.Layout(width='300px'),
)
status_output = widgets.Output()
latex_output = widgets.Textarea(
    description='LaTeX', layout=widgets.Layout(width='100%', height='600px'),
)

def refresh_latex(_=None):
    with status_output:
        status_output.clear_output(wait=True)
        if not dataset_widget.value or not fit_widget.value:
            print('Select at least one dataset and one fit.')
            return
        try:
            latex, messages = generate_latex(
                dataset_widget.value, fit_widget.value, central_widget.value,
                precision_widget.value, covariance_precision_widget.value,
                full_precision_widget.value,
                burn_in_widget.value, thin_widget.value,
            )
        except Exception as error:
            print(f'{type(error).__name__}: {error}')
            raise
        latex_output.value = latex
        print('\n'.join(messages))
        print(f'\nGenerated {len(latex):,} LaTeX characters. Copy from the box below.')

generate_button.on_click(refresh_latex)
controls = widgets.VBox([
    widgets.HBox([dataset_widget, fit_widget]),
    central_widget,
    widgets.HBox([
        precision_widget, covariance_precision_widget, full_precision_widget,
    ]),
    widgets.HBox([burn_in_widget, thin_widget]),
    generate_button, status_output, latex_output,
])
display(controls)

# Populate all three datasets on the first run.
refresh_latex()